In [7]:
import pandas as pd

airports = pd.read_csv(
    "Data/OurAirports/airports.csv",
    low_memory=False
)

print(airports.shape)
airports.head()

(85558, 19)


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [8]:
us_airports = airports[
    (airports["iso_country"] == "US") &
    (airports["iata_code"].notna())
]

print("Số sân bay Mỹ có IATA:")
print(len(us_airports))

us_airports[
    [
        "iata_code",
        "name",
        "municipality",
        "latitude_deg",
        "longitude_deg"
    ]
].head()

Số sân bay Mỹ có IATA:
2035


,iata_code,name,municipality,latitude_deg,longitude_deg
411,OCA,Ocean Reef Club Airport,Key Largo,25.325399,-80.274803
634,CSE,Crested Butte Airpark,Crested Butte,38.851918,-106.928341
891,CUS,Columbus Airport,Columbus,31.823898,-107.629924
988,JCY,LBJ Ranch Airport,Stonewall,30.251801,-98.622498
1288,WLR,Loring Seaplane Base,Loring,55.601299,-131.636993


In [10]:
airport_list = [
    "ATL",
    "LAX",
    "ORD",
    "DFW",
    "DEN",
    "JFK",
    "SFO",
    "LAS",
    "MCO",
    "CLT"
]

target_airports = us_airports[
    us_airports["iata_code"].isin(
        airport_list
    )
]

target_airports[
    [
        "iata_code",
        "name",
        "latitude_deg",
        "longitude_deg"
    ]
]

,iata_code,name,latitude_deg,longitude_deg
38607,ATL,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101
38898,CLT,Charlotte Douglas International Airport,35.214001,-80.943100
39040,DEN,Denver International Airport,39.860027,-104.673792
39045,DFW,Dallas Fort Worth International Airport,32.896801,-97.038002
40164,JFK,John F. Kennedy International Airport,40.639447,-73.779317
40272,LAS,Harry Reid International Airport,36.083361,-115.151817
40274,LAX,Los Angeles International Airport,33.942501,-118.407997
40479,MCO,Orlando International Airport,28.429399,-81.308998
40846,ORD,Chicago O'Hare International Airport,41.978600,-87.904800
42447,SFO,San Francisco International Airport,37.619806,-122.374821


In [11]:
airport_list = [
    "ATL",
    "LAX",
    "ORD",
    "DFW",
    "DEN",
    "JFK",
    "SFO",
    "LAS",
    "MCO",
    "CLT"
]

target_airports = us_airports[
    us_airports["iata_code"].isin(
        airport_list
    )
]

target_airports[
    [
        "iata_code",
        "name",
        "latitude_deg",
        "longitude_deg"
    ]
]

,iata_code,name,latitude_deg,longitude_deg
38607,ATL,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101
38898,CLT,Charlotte Douglas International Airport,35.214001,-80.943100
39040,DEN,Denver International Airport,39.860027,-104.673792
39045,DFW,Dallas Fort Worth International Airport,32.896801,-97.038002
40164,JFK,John F. Kennedy International Airport,40.639447,-73.779317
40272,LAS,Harry Reid International Airport,36.083361,-115.151817
40274,LAX,Los Angeles International Airport,33.942501,-118.407997
40479,MCO,Orlando International Airport,28.429399,-81.308998
40846,ORD,Chicago O'Hare International Airport,41.978600,-87.904800
42447,SFO,San Francisco International Airport,37.619806,-122.374821


In [14]:
import duckdb

top50 = duckdb.sql("""
WITH airports AS (

    SELECT Origin AS airport
    FROM read_parquet(
        'Data/Flights/us_flights_2024_all.parquet'
    )

    UNION ALL

    SELECT Dest AS airport
    FROM read_parquet(
        'Data/Flights/us_flights_2024_all.parquet'
    )

)

SELECT
    airport,
    COUNT(*) AS total_movements
FROM airports
GROUP BY airport
ORDER BY total_movements DESC
LIMIT 50
""").df()

top50.head()

,airport,total_movements
0,ATL,683735
1,DFW,627150
2,DEN,617254
3,ORD,560061
4,CLT,435100


In [15]:
import pandas as pd

airports = pd.read_csv(
    "Data/OurAirports/airports.csv",
    low_memory=False
)

target_airports = top50.merge(
    airports,
    left_on="airport",
    right_on="iata_code",
    how="left"
)

In [16]:
target_airports = target_airports[
    [
        "airport",
        "name",
        "municipality",
        "iso_region",
        "latitude_deg",
        "longitude_deg",
        "total_movements"
    ]
]

target_airports.head()

,airport,name,municipality,iso_region,latitude_deg,longitude_deg,total_movements
0,ATL,Hartsfield Jackson Atlanta International Airport,Atlanta,US-GA,33.636700,-84.428101,683735
1,DFW,Dallas Fort Worth International Airport,Dallas-Fort Worth,US-TX,32.896801,-97.038002,627150
2,DEN,Denver International Airport,Denver,US-CO,39.860027,-104.673792,617254
3,ORD,Chicago O'Hare International Airport,Chicago,US-IL,41.978600,-87.904800,560061
4,CLT,Charlotte Douglas International Airport,Charlotte,US-NC,35.214001,-80.943100,435100


In [18]:
target_airports.to_csv(
    "Data/OpenWeather/target_airports_top50.csv",
    index=False
)

print("Saved!")

Saved!
